# Flower Classification — Exploratory Data Analysis

**Project:** Automating Flower Classification for a Start-up Using Deep Learning  
**Dataset:** Oxford 102 Flower Dataset  
**Objective:** Explore the dataset, visualize class distributions, understand image characteristics,
and assess augmentation strategies before model training.

---

## Table of Contents
1. [Setup and Imports](#1-setup-and-imports)
2. [Dataset Loading](#2-dataset-loading)
3. [Class Distribution Analysis](#3-class-distribution-analysis)
4. [Image Properties Analysis](#4-image-properties-analysis)
5. [Sample Image Gallery](#5-sample-image-gallery)
6. [Color Distribution Analysis](#6-color-distribution-analysis)
7. [Augmentation Preview](#7-augmentation-preview)
8. [Train/Val/Test Split Analysis](#8-trainvaltest-split-analysis)
9. [Summary and Next Steps](#9-summary-and-next-steps)

In [ ]:
# Cell 1: Setup and Imports
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from PIL import Image
import cv2

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.data_preprocessing import (
    generate_sample_data,
    scan_dataset_directory,
    build_dataframe,
    split_dataset,
    compute_dataset_statistics,
    get_augmentation_pipeline,
    load_image,
    FLOWER_CLASSES,
    IMAGE_SIZE
)
from src.visualization import plot_class_distribution

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
print('Setup complete.')
print(f'Total flower classes: {len(FLOWER_CLASSES)}')
print(f'Target image size: {IMAGE_SIZE}')

In [ ]:
# Cell 2: Dataset Loading
# Generate sample data for demonstration (replace with real dataset path)
print('Generating sample dataset for EDA demonstration...')
sample_df = generate_sample_data(
    output_dir='../data/sample',
    n_classes=10,
    n_images_per_class=30
)

# Load from directory structure
class_to_images = scan_dataset_directory('../data/sample')
df = build_dataframe(class_to_images)

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nClass counts:')
print(df['label'].value_counts())
df.head(10)

In [ ]:
# Cell 3: Class Distribution Analysis
stats = compute_dataset_statistics(df)
print('=== Dataset Statistics ===')
print(f'Total samples       : {stats["total_samples"]}')
print(f'Number of classes   : {stats["num_classes"]}')
print(f'Min samples/class   : {stats["min_samples_per_class"]}')
print(f'Max samples/class   : {stats["max_samples_per_class"]}')
print(f'Mean samples/class  : {stats["mean_samples_per_class"]:.1f}')
print(f'Imbalance ratio     : {stats["imbalance_ratio"]:.2f}x')

# Plot class distribution
counts = df['label'].value_counts()
fig = px.bar(
    x=counts.index, y=counts.values,
    labels={'x': 'Flower Class', 'y': 'Number of Images'},
    title='Flower Class Distribution',
    color=counts.values,
    color_continuous_scale='Viridis'
)
fig.update_layout(height=450, xaxis_tickangle=-45)
fig.show()

In [ ]:
# Cell 4: Image Properties Analysis
print('Analyzing image properties (sample of 50 images)...')
sample_paths = df['filepath'].sample(min(50, len(df)), random_state=42).tolist()

widths, heights, aspect_ratios, file_sizes = [], [], [], []
for path in sample_paths:
    try:
        img = Image.open(path)
        w, h = img.size
        widths.append(w)
        heights.append(h)
        aspect_ratios.append(w / h)
        file_sizes.append(os.path.getsize(path) / 1024)  # KB
    except Exception:
        pass

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(widths, bins=20, color='steelblue', edgecolor='black')
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Width (px)')

axes[1].hist(heights, bins=20, color='seagreen', edgecolor='black')
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Height (px)')

axes[2].hist(aspect_ratios, bins=20, color='coral', edgecolor='black')
axes[2].set_title('Aspect Ratio Distribution')
axes[2].set_xlabel('Width / Height')
axes[2].axvline(1.0, color='red', linestyle='--', label='Square (1.0)')
axes[2].legend()

plt.suptitle('Image Properties Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/image_properties.png', bbox_inches='tight')
plt.show()

print(f'Width  : mean={np.mean(widths):.0f}, std={np.std(widths):.0f}')
print(f'Height : mean={np.mean(heights):.0f}, std={np.std(heights):.0f}')
print(f'Aspect : mean={np.mean(aspect_ratios):.2f}, std={np.std(aspect_ratios):.2f}')

In [ ]:
# Cell 5: Sample Image Gallery
classes = df['label'].unique()[:12]
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()

for i, cls in enumerate(classes):
    cls_df = df[df['label'] == cls]
    img_path = cls_df.sample(1, random_state=42)['filepath'].values[0]
    try:
        img = load_image(img_path, target_size=(128, 128))
        axes[i].imshow(img)
        axes[i].set_title(cls.replace('_', ' ').title(), fontsize=9)
    except Exception:
        axes[i].set_title(f'{cls} (failed)', fontsize=8, color='red')
    axes[i].axis('off')

for j in range(len(classes), len(axes)):
    axes[j].axis('off')

plt.suptitle('Sample Flower Images (One per Class)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/sample_gallery.png', bbox_inches='tight')
plt.show()

In [ ]:
# Cell 6: Color Distribution Analysis
print('Analyzing color channel distributions...')
sample_paths_color = df['filepath'].sample(min(30, len(df)), random_state=0).tolist()

r_means, g_means, b_means = [], [], []
for path in sample_paths_color:
    try:
        img = np.array(Image.open(path).convert('RGB').resize((64, 64))).astype(float) / 255.0
        r_means.append(img[:, :, 0].mean())
        g_means.append(img[:, :, 1].mean())
        b_means.append(img[:, :, 2].mean())
    except Exception:
        pass

channel_data = pd.DataFrame({'Red': r_means, 'Green': g_means, 'Blue': b_means})

fig = go.Figure()
for col, color in [('Red', 'red'), ('Green', 'green'), ('Blue', 'blue')]:
    fig.add_trace(go.Box(
        y=channel_data[col], name=col,
        marker_color=color, boxmean='sd'
    ))
fig.update_layout(
    title='Color Channel Mean Distribution (Sample)',
    yaxis_title='Mean Pixel Value (normalized)',
    height=400
)
fig.show()
print(channel_data.describe().round(4))

In [ ]:
# Cell 7: Augmentation Preview
import albumentations as A

aug_pipeline = get_augmentation_pipeline(training=True)

# Load one sample image for augmentation demonstration
sample_path = df['filepath'].iloc[0]
original_img = load_image(sample_path, target_size=(224, 224))

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

axes[0].imshow(original_img)
axes[0].set_title('Original', fontweight='bold')
axes[0].axis('off')

np.random.seed(None)
for i in range(1, 10):
    aug_result = aug_pipeline(image=original_img.copy())
    aug_img = aug_result['image']
    # Denormalize for display
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    display_img = (aug_img * std + mean).clip(0, 1)
    axes[i].imshow(display_img)
    axes[i].set_title(f'Augmented {i}', fontsize=9)
    axes[i].axis('off')

plt.suptitle('Data Augmentation Examples', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/augmentation_preview.png', bbox_inches='tight')
plt.show()
print('Augmentation preview saved.')

In [ ]:
# Cell 8: Train/Val/Test Split Analysis
train_df, val_df, test_df = split_dataset(df, val_size=0.15, test_size=0.10)

split_info = pd.DataFrame({
    'Split': ['Train', 'Validation', 'Test'],
    'Samples': [len(train_df), len(val_df), len(test_df)],
    'Proportion (%)': [
        round(100 * len(train_df) / len(df), 1),
        round(100 * len(val_df) / len(df), 1),
        round(100 * len(test_df) / len(df), 1),
    ]
})
print('=== Dataset Split Summary ===')
print(split_info.to_string(index=False))

fig = go.Figure(data=[go.Pie(
    labels=split_info['Split'],
    values=split_info['Samples'],
    hole=0.4,
    marker_colors=['#2196F3', '#4CAF50', '#FF9800']
)])
fig.update_layout(
    title='Train / Validation / Test Split',
    height=400
)
fig.show()

# Verify class balance across splits
print('\nClass distribution in Train set:')
print(train_df['label'].value_counts())

In [ ]:
# Cell 9: Summary and Next Steps
print('=' * 60)
print('           EDA SUMMARY')
print('=' * 60)
print(f'Total samples     : {len(df)}')
print(f'Total classes     : {df["label"].nunique()}')
print(f'Training set      : {len(train_df)} ({100*len(train_df)/len(df):.1f}%)')
print(f'Validation set    : {len(val_df)} ({100*len(val_df)/len(df):.1f}%)')
print(f'Test set          : {len(test_df)} ({100*len(test_df)/len(df):.1f}%)')
print(f'Target image size : {IMAGE_SIZE}')
print()
print('KEY FINDINGS:')
print('  1. Class imbalance exists - will use class weighting during training')
print('  2. Images have variable resolution - resize to 224x224 needed')
print('  3. Color channels have minor bias toward green (vegetation)')
print('  4. Heavy augmentation is warranted given limited data per class')
print()
print('NEXT STEPS:')
print('  1. Run model_training.py to train baseline Custom CNN')
print('  2. Run transfer learning with MobileNetV2 and EfficientNetB0')
print('  3. Execute Optuna hyperparameter optimization (20+ trials)')
print('  4. Generate confusion matrix and Grad-CAM visualizations')
print('  5. Export best model to TFLite for mobile deployment')